# Visualising data

Plots of devices and tests: time series, scatter plots, heatmaps and box plots with matplotlib, fast interactive time series with uPlot, and maps. Interactive panels (bokeh) are in [19_interactive_plots_with_panel.ipynb](19_interactive_plots_with_panel.ipynb), and more uPlot options in [18_interactive_plots.ipynb](18_interactive_plots.ipynb).

Plots need the plotting libraries, an optional extra. Without them, a plot raises an error that says how to install them:

```
pip install "scdata[plot]"
```

In [ ]:
import logging

import scdata as sc
import smartcitizen_connector
from scdata.tools.custom_logger import set_logger_level

# Only errors: 'WARNING', 'INFO' or 'DEBUG' show what scdata and smartcitizen-connector do
set_logger_level('ERROR')
logging.getLogger('smartcitizen_connector').setLevel('ERROR')

sc.__version__

In [ ]:
import pandas as pd

start = (pd.Timestamp.now(tz='UTC') - pd.Timedelta('2D')).strftime('%Y-%m-%d')
test = sc.Test(name='EXAMPLES_PLOTS',
               devices=[sc.Device(params=sc.APIParams(id=device_id)) for device_id in [18154, 17592]],
               force_recreate=True)
for device in test.devices:
    device.options.min_date = start
await test.load()
test.process()

## How plots are described

All plots take:

- `traces`: what to plot. Each trace has the `devices` (a list of ids, or `"all"`), the `channel` and, for time series, the `subplot`
- `options`: how to prepare the data (`frequency` to resample, `min_date`, `max_date`, `clean_na`)
- `formatting`: titles, labels, ranges, sizes

Plots of a device (`device.ts_plot(...)`) do not need `devices` in the traces.

## Time series

### matplotlib

In [ ]:
traces = {
    '1': {'devices': 'all', 'channel': 'NOISE_A', 'subplot': 1},
    '2': {'devices': 'all', 'channel': 'TEMP', 'subplot': 2},
    '3': {'devices': 'all', 'channel': 'SEN5X_PM_25', 'subplot': 3},
}

options = {'frequency': '10Min'}

formatting = {
    'title': 'Noise, temperature and PM2.5',
    'ylabel': {1: 'Noise (dBA)', 2: 'Temperature (degC)', 3: 'PM2.5 (ug/m3)'},
    'yrange': {1: [30, 80], 2: [10, 35], 3: [0, 50]},
    'sharex': True,
    'height': 10,
    'width': 15,
}

figure = test.ts_plot(traces=traces, options=options, formatting=formatting)

Extras (`mean`, `max`, `min`) summarise the devices of a trace, and decorators mark dates (`xtext` needs a `yrange`):

In [ ]:
traces = {'1': {'devices': 'all', 'channel': 'TEMP', 'subplot': 1, 'extras': ['mean', 'max', 'min']}}

middle = (pd.Timestamp.now(tz='UTC') - pd.Timedelta('1D')).strftime('%Y-%m-%d')
formatting = {'title': 'Temperature', 'ylabel': {1: 'Temperature (degC)'}, 'yrange': {1: [10, 35]},
              'decorators': {'axvline': [middle], 'xtext': {middle: 'Yesterday'}}}

figure = test.ts_plot(traces=traces, options={'frequency': '30Min'}, formatting=formatting)

### uPlot

Interactive and fast, also with many points (drag to zoom, double click to reset):

In [ ]:
traces = {
    '1': {'devices': 'all', 'channel': 'NOISE_A', 'subplot': 1},
    '2': {'devices': 'all', 'channel': 'NO2', 'subplot': 2},
}

test.ts_uplot(traces=traces,
              formatting={'width': 900, 'height': 300, 'padding-bottom': 600, 'title': 'Noise and NO2',
                          'ylabel': {1: 'Noise (dBA)', 2: 'NO2 (ppb)'}},
              options={'frequency': '5Min'})

A single device:

In [ ]:
test.get_device(18154).ts_uplot(traces={'1': {'channel': 'HUM', 'subplot': 1}},
                                   formatting={'width': 900, 'height': 300, 'title': 'Humidity', 'ylabel': {1: 'Humidity (%rh)'}},
                                   options={'frequency': '1Min'})

## Scatter plots

In [ ]:
traces = {
    '1': {'devices': 18154, 'channel': 'TEMP'},
    '2': {'devices': 18154, 'channel': 'SEN5X_TEMP'},
}

figure = test.scatter_plot(traces=traces, options={'frequency': '10Min'},
                           formatting={'title': 'SHT vs SEN5X temperature', 'height': 8})

`ts_scatter` shows both series in time and their scatter plot:

In [ ]:
traces = {
    '1': {'devices': 18154, 'channel': 'SEN5X_PM_25'},
    '2': {'devices': 17592, 'channel': 'SEN5X_PM_25'},
}

figure = test.ts_scatter(traces=traces, options={'frequency': '30Min'},
                         formatting={'title': 'PM2.5 of both devices', 'ylabel': 'PM2.5 (ug/m3)'})

## Heatmap

Values by hour of the day:

In [ ]:
figure = test.heatmap_plot(traces={'1': {'devices': 18154, 'channel': 'NOISE_A'}},
                           options={'frequency': '10Min'},
                           formatting={'title': 'Noise by hour'})

## Box plot

Distributions by period (`frequency`):

In [ ]:
figure = test.box_plot(traces={'1': {'devices': 'all', 'channel': 'NOISE_A'}},
                       options={'frequency': '1h'},
                       formatting={'title': 'Noise by hour'})

## Maps

Maps with [folium](https://python-visualization.github.io/folium/). `device_metric_map` places the devices at their location, colored by the average of a channel in each period, with a time slider:

In [ ]:
test.device_metric_map('SEN5X_PM_25', options={'period': '6h', 'location': 'average', 'zoom': 4, 'minmax': True})

`path_plot` draws the path of devices with GPS (`GPS_LAT`, `GPS_LONG`), colored by a channel. These devices are fixed, so here we give one of them a made up path:

In [ ]:
import numpy as np

moving = test.get_device(18154)
path = moving.data.index[-120:]
moving.data.loc[path, 'GPS_LAT'] = np.linspace(41.380, 41.400, len(path))
moving.data.loc[path, 'GPS_LONG'] = np.linspace(2.170, 2.195, len(path))

moving.path_plot(channel='NOISE_A', map_type='static', start_date=str(path[0]),
                 options={'period': '5Min', 'location': 'average', 'zoom': 14, 'minmax': True})

## Saving figures

matplotlib plots return the figure:

In [ ]:
from os.path import join
from scdata._config import config

figure = test.ts_plot(traces={'1': {'devices': 'all', 'channel': 'TEMP', 'subplot': 1}}, options={'frequency': '10Min'})
figure.savefig(join(config.paths['export'], 'examples', 'temperature.png'), dpi=150, bbox_inches='tight')

## Your own plots

The data of each device is a `pandas.DataFrame`, so any plotting library works:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
for device in test.devices:
    device.data['NOISE_A'].resample('1h').mean().plot(ax=ax, label=device.id)
ax.set_ylabel('Noise (dBA)')
ax.legend();